# mesh01 — 표적 10종을 무엇으로 짓고, 무엇으로 채점하나 (전체 지도)

> ⚠ **이 노트북은 생성물이다.** 수정은 `report_mesh/src/make_mesh01.py` 에서 하고
> 재실행할 것(`.ipynb` 를 직접 고치면 다음 빌드에서 사라진다).

**이 편이 답하는 질문** — 우리가 레이더 시뮬레이션에 쓰는 드론 표적 10종은 무엇으로 만들어졌고, 그 형상을 무엇이 채점하는가.

**무엇을 근거로 하는가** (본문 수치는 손으로 적지 않고 아래 원장에서 주입한다)

| 원장 | 무엇이 들어 있나 |
|---|---|
| `report_mesh/outputs/mesh_verify.json` | 기하 검증 스위트 A~I — 이 시리즈의 기본 원장 |
| `outputs/mesh_inspect_body_arms_0816.json` | 동체·팔·다리·모터 벨 전수 실측(10종) + matrice4e 공식 CAD 정정 착지 검증 |
| `outputs/mesh_inspect_gimbal_sensors_0816.json` | 짐벌·카메라·센서 검사(10종) — 부착 게이트 A~D |
| `outputs/mesh_internal_metal_check_0816.json` | «내부 금속이 정말 셸 안에 있나» 기체별 판정(PASS/FAIL/N/A/UNKNOWN) |
| `outputs/mesh_inspect_materials_check_0816.json` | 재질 배정·검사기·프롭 외 부품 감사(13그룹·10종) |
| `outputs/meshfix_matrice4e.json` | DJI 공식 STEP 대조 정정 명세 14건(matrice4e) |
| `docs/MESH_AUDIT_0816.md` | 적대적 감사 — 발견·반증·수리 우선순위(§⑤) |
| `assets/meshes/reference/SOURCES.md` | 참조 CAD·스캔의 출처와 라이선스 |

**한 줄 요약** — 인터넷의 드론 3D 모델은 시각용 껍데기라 레이더 시뮬레이션에 못 쓴다.
그래서 제조사 공식 제원과 (있는 경우) 공식 CAD 로부터 코드가 드론 10종을 직접 깎는다
(총 287,907개 삼각형 · 부품 304개). 이 편은 그 **전체 지도**다 —
자료가 어디까지 제작에 들어갔고, 어디서부터 독립 채점이 시작되는지.

이 시리즈(mesh01~08)는 **파이썬 기초만 아는 독자**를 위한 3D 모델 제작 가이드다.
본문의 모든 숫자는 손으로 적지 않고 위 원장에서 자동 주입했으며,
**모든 사실 옆에 `← 출처:` 를 단다** — 어느 파일·어느 측정에서 온 정보인지 추적할 수 있게.

## 용어풀이 (이 리포트에 나오는 순서대로)

| 용어 | 한 줄 풀이 |
|---|---|
| **메쉬(mesh)** | 작은 삼각형 수천 장을 이어붙여 만든 3D 표면. 컴퓨터가 형상을 저장하는 가장 보편적 방식 |
| **꼭짓점(vertex)** | 3D 공간의 점 (x, y, z). 삼각형의 모서리 끝점 |
| **면(face)** | 꼭짓점 3개를 이은 삼각형 1장. 메쉬의 최소 단위 |
| **법선(normal)** | 삼각형이 바라보는 방향(수직 화살표). 물체의 '겉'과 '속'을 구분한다 |
| **수밀(watertight)** | 구멍이 하나도 없는 닫힌 표면 — 물을 부어도 새지 않는 그릇 |
| **경계 모서리** | 삼각형 **하나만** 쓰는 모서리. 구멍의 테두리다 |
| **그룹(group)** | 면마다 붙인 부위 이름표(body/prop/motor …). 부위별 재질 배정의 열쇠 |
| **OBJ** | 꼭짓점·삼각형 목록을 적는 텍스트 3D 파일 형식. Sionna/Mitsuba 가 바로 읽는다 |
| **파라메트릭 CAD** | 치수(파라미터)를 넣으면 코드가 형상을 만들어 주는 설계 방식 |
| **불리언(boolean union)** | 두 입체를 '합집합' 으로 녹여 붙여 내부의 숨은 면을 없애는 연산 |
| **축간거리(wheelbase)** | 마주 보는 두 로터 축 사이의 거리. **메쉬에서 잰 값** |
| **대각(diagonal)** | 제조사가 공표하는 대각 치수. 축간거리와 **같지 않을 수 있다**(§1.3) |
| **RCS(σ)** | 레이더에게 물체가 얼마나 '밝게' 보이는지의 면적값 [m²] |
| **PO / SBR** | 물리광학 / 광선발사·반사 — 표면에서 RCS 를 계산하는 두 방법 |
| **\|Γ\|(반사계수)** | 전파가 재질 표면에서 반사되는 진폭 비율. 금속≈1.0, 플라스틱≈0.3 |
| **평판극한 상한** | 형상 차이를 «같은 크기 평판이면 최대 이만큼» 으로 옮긴 **상한값**. 커널 계산 결과가 아니라 크기 감각을 주는 자다 |
| **파장 λ** | 전파의 한 주기 길이. 메쉬 삼각형은 λ 보다 충분히 작아야 형상을 '전파의 눈'으로 담는다 |

## 0. 이 편이 답하는 질문

이 저장소는 통신 신호를 조명 삼아 드론을 탐지하는 시뮬레이터다. 시나리오는 하나가 아니다 —
**패시브 바이스태틱**(남의 신호를 빌려 쓴다)과 **모노스태틱**(파형을 우리가 안다)을 함께 본다
← 출처: `README.md`. 그 시뮬레이션의 **표적**이 되는
드론 10종(Mini 5 Pro, Mavic 4 Pro, Matrice 4E, S1000+, Phantom 4, Typhoon H (H480), X500 V2, Phantom 3 Professional, Matrice 350 RTK, Mini 2)의
3D 모델을 어떻게 만들었는지가 이 시리즈의 주제다.

⭐ **표적 축은 시나리오와 무관하다.** σ·마이크로도플러·앙각은 «누가 신호를 쐈나» 와 상관없이
같은 형상에서 나온다. 그래서 이 시리즈는 시나리오를 안 고르고 형상만 다룬다.

이 편(mesh01)은 다섯 가지 질문에 답한다:

1. **삼각형 메쉬가 뭔가?** — 종이접기 비유로 (§1)
2. **왜 인터넷 3D 모델을 안 받고 코드로 만들었나?** (§2)
3. **그럼 참조 자료는 어디까지 제작에 들어갔나?** (§2.3 — 이 편에서 가장 자주 오해받는 자리)
4. **'OBJ 1개 = 부위 1개 = 재질 1개' 는 무슨 뜻인가?** (§3)
5. **자료 → 제작 → 검사기 → 원장, 전체 지도는 어떻게 생겼나?** (§4)

세부(몸체 깎는 법, 프로펠러, 자료 출처, 재질, 기하 품질, 실물 대조)는
mesh02~08 각 편이 하나씩 맡는다 — 목차는 §5.

## 1. 삼각형 메쉬란 무엇인가 — 종이접기 다면체

**비유**: 축구공 모양 종이 다면체를 접어 본 적이 있는가? 평평한 종이 조각(오각형·육각형)을
수십 장 이어붙이면 거의 둥근 공이 된다. 조각을 잘게 쓸수록 더 매끈해진다.
3D 메쉬가 정확히 그것이다 — 다만 조각이 전부 **삼각형**이고, 종이 대신 숫자로 접는다.

왜 하필 삼각형인가? **점 3개는 반드시 한 평면 위에 있기 때문**이다. 사각형부터는 뒤틀릴 수
있어(네 점이 한 평면에 안 놓임) 계산이 모호해진다. 그래서 그래픽스·전파 시뮬레이션 모두
삼각형을 최소 단위로 쓴다.

우리 코드의 메쉬 정의는 놀랄 만큼 단순하다:

```
핵심 개념 — 메쉬(Mesh)는 딱 두 가지로 이루어집니다
  1) 꼭짓점(vertex) 목록 : 3D 점 (x, y, z) 들의 리스트
  2) 면(face) 목록       : "몇 번 꼭짓점 3개를 이어 삼각형을 만들지"
거기에 우리는 "그룹(group)" 하나를 더 붙입니다. 면마다 어떤 **재질 그룹**
(예: body / arm / motor / prop / absorber ...)에 속하는지 이름표를 답니다.
```

← 출처: `src/geom.py` 15~21행 모듈 docstring (그대로 인용).
실제 클래스도 딱 세 줄이다 — `.v`(꼭짓점), `.f`(삼각형 인덱스), `.g`(면별 그룹 이름)
← 출처: `src/geom.py:41` `class Mesh` docstring.

### 1.2 법선과 watertight — '겉면 스티커'와 '물 안 새는 그릇'

종이 다면체를 접을 때 겉과 속을 뒤집어 붙이면 이상해진다. 메쉬도 같다. 삼각형마다
**법선(normal)** — 어느 쪽이 '겉'인지 가리키는 수직 화살표 — 이 있고, 모든 법선이
바깥을 향해야 전파 시뮬레이터가 "여기가 물체 표면" 을 올바로 인식한다.
우리 PO 커널의 조명 판정이 `n̂·û > 0` 이라, 법선이 뒤집힌 면은 조명 여부가 **반대로** 정해진다
← 출처: `src/rcs_po.py` 조명 판정.

**watertight(수밀)** 는 표면에 구멍이 하나도 없다는 뜻이다 — 그릇에 물을 부어도 안 샌다.
구멍이 있으면 (a) 부피를 정의할 수 없고 (b) **안/밖 판정(`contains`)이 정의되지 않는다.**
두 번째가 실질적인 피해다: 내부 판정을 쓰는 검사가 그 부품을 조용히 건너뛴다.

두 개념을 여기서는 **뜻만** 익히고, 지금 우리 메쉬의 성적표는 §4.3~§4.4 에서
검사기·예산과 함께 읽는다 — 숫자 하나로 «통과» 를 말하는 것이 정확하지 않기 때문이다.

### 1.3 표적 10종 — 한 표로

| 드론 | 근거 | 축간거리[mm] | 프롭Ø[mm]×수 | 꼭짓점 | 삼각형 | 그룹 | 수밀 | 지금 선언된 결함 |
|---|---|---|---|---|---|---|---|---|
| Mini 5 Pro | **[B]** | 248.3 | 152×4 | 14,564 | 29,036 | 9 | 23/23 | 세로 배율 1.2985(전 부품 늘림) |
| Mavic 4 Pro | **[C]** | 441.0 | 267×4 | 15,006 | 29,932 | 8 | 20/20 | 세로 배율 1.3524 · 짐벌이 발보다 15.35 mm 아래 |
| Matrice 4E | **[A]** | 438.9 | 274×4 | 15,387 | 30,662 | 9 | 28/28 | 로터면이 CAD 보다 18.5 mm 위(F19~F21 보류) |
| S1000+ | **[B]** | 1,043.5 | 381×8 | 17,755 | 35,374 | 10 | 49/49 | 카본 센터플레이트가 `plastic` 그룹 |
| Phantom 4 | **[B]** | 356.9 | 240×4 | 14,343 | 28,594 | 8 | 23/23 | L/W 강제 → 축간거리 +1.98 % · 착륙아치 8.3~8.5 mm 뜸 |
| Typhoon H (H480) | **[A]** | 480.0 | 230×6 | 17,424 | 34,732 | 9 | 29/29 | — |
| X500 V2 | **[A]** | 500.0 | 254×4 | 9,615 | 19,030 | 10 | 50/50 | 레일 4.0 mm 뜸 · accent↔arm 동일평면 |
| Phantom 3 Professional | **[B]** | 350.0 | 240×4 | 14,296 | 28,508 | 8 | 21/21 | 착륙아치 13.7~13.8 mm 뜸 · 짐벌 3조각이 떨어져 있음 |
| Matrice 350 RTK | **[B]** | 895.0 | 533×4 | 13,194 | 26,216 | 9 | 43/43 | 프롭 허브가 벨 위 6.0 mm 뜸 |
| Mini 2 | **[A]** | 213.0 | 119×4 | 12,948 | 25,823 | 8 | 17/18 | 셸에 삼각형 1장 구멍(경계 모서리 3) |
| **합계** | | | | **144,532** | **287,907** | | **303/304** | |

**«축간거리» 열을 쓰는 이유** — 이 열은 마주 보는 두 로터 축 사이 거리를 **메쉬에서 잰 값**이다.
제조사가 공표하는 «대각» 과 같지 않을 수 있다. 가장 큰 차이는 Mini 5 Pro 로,
공표 대각 275 mm ↔ 축간거리
248.26 mm 다. 로터가 정사각형이 아니라
사다리꼴로 놓이기 때문이고, 이것은 결함이 아니라 **선언된 선택**이다
← 출처: `src/drones.py` mini5pro `note`(«diagonal_mm 은 로터 위치를 정하지 않는다»)·
`outputs/mesh_inspect_body_arms_0816.json` `per_drone.mini5pro.wheelbase_mm`.

⚠ **리포트·발표에서 Mini 5 Pro 의 로터 간격을 쓸 때는 축간거리를 인용할 것.**
«대각 275 mm» 를 로터 간격으로 쓰면 9.7 % 틀린다.

**근거 등급** — 형상이 어디서 왔는가:

| 등급 | 뜻 | 이 저장소에서의 실체 |
|---|---|---|
| **[A]** | 공식 CAD 직접 | 제조사가 낸 CAD 파일을 열어 잰 값 — DJI Matrice 4T STEP · DJI Mini 2 GLB · Holybro X500 v2 STEP · Yuneec Typhoon H480 실물 CAD |
| **[B]** | 사진 계측 | 제품사진·매뉴얼 도해를 픽셀로 잰 값 |
| **[C]** | 계열 유추 | 다른 기체의 실측을 크기비로 옮긴 값 |
| **[D]** | 대리 | 다른 제조사의 부품을 대신 세운 값 |

⚠ 등급은 **«공표 숫자만으로는 안 정해지는 형상»** 이 어디서 왔는가를 매긴다. 공표 제원(외형 L×W×H · 프로펠러 지름 · 대각)은 10종 공통 입력이라 이 축 밖이다 — 그 숫자들의 출처는 mesh03 이 따로 맡는다.

| 드론 | 등급 | 근거 |
|---|---|---|
| Mini 5 Pro | **[B]** | 제품사진 계측. ⚠ 셸 높이의 1차 출처가 없다 — 공표 91 mm 가 프롭을 포함해 셸을 구속하지 않는다 |
| Mavic 4 Pro | **[C]** | 암 폭은 **Mini 5 Pro 실측의 크기비 이전**. DJI 가 Mavic 4 Pro CAD 를 공개하지 않는다 |
| Matrice 4E | **[A]** | DJI Matrice 4T 공식 STEP 로 형상 상수 14건 정정, 전부 착지 (`meshfix_matrice4e.json` · `body_arms.meshfix_matrice4e_landed`). ⚠ 짐벌·카메라 블록만 4T↔4E 가 달라 **치수는 사진, 매다는 자리만 CAD** |
| S1000+ | **[B]** | 공표 제원(카본 튜브 25 mm) + 제품사진 |
| Phantom 4 | **[B]** | 공표 제원 + 제품사진. 실기체 3D 스캔(CC-BY)은 **채점 전용**이라 제작에 안 썼다 |
| Typhoon H (H480) | **[A]** | Yuneec 실물 CAD(ethz-asl/rotors_simulator, Apache-2.0) — 암 12.0×12.6 ↔ CAD 12.002 |
| X500 V2 | **[A]** | Holybro 공식 STEP 프레임(암 단면 16.0×16.0 mm 재현) |
| Phantom 3 Professional | **[B]** | 공표 제원 + 제품사진 |
| Matrice 350 RTK | **[B]** | 공표 제원 + 제품사진(암 튜브 22 mm) |
| Mini 2 | **[A]** | DJI 공식 GLB(WM161) 실측 — 셸 6 스테이션 중 가운데 4개가 GLB 와 0.5 % 안 |

← 출처: `outputs/mesh_inspect_body_arms_0816.json`(암 단면·셸 스테이션 실측 대조)·
`outputs/meshfix_matrice4e.json`(공식 STEP)·`assets/meshes/reference/SOURCES.md`.

### 1.4 눈으로 보기 — 대표 두 기종

대표를 **둘** 세운다. 두 기체가 다른 것을 대표하기 때문이다.

- **Mini 5 Pro** — 실측 캠페인의 표적이다 ← 출처: `README.md` 실측 계획.
  형상 근거는 사진 계측이고, **셸 높이의 1차 출처가 없다**는 한계를 그대로 안고 있다(§4.4).
- **Matrice 4E** — 공식 CAD 대조가 끝난 기체다. DJI Matrice 4T 공식 STEP 으로
  형상 상수 14건을 정정했고 **14/14 착지이다.
  ← 출처: `outputs/mesh_inspect_body_arms_0816.json` `meshfix_matrice4e_landed`.

![wireframe mini5pro](outputs/figures/wireframe_mini5pro.png)

**그림 1** — Mini 5 Pro 메쉬 3면 ← 그림 생성:
`report_mesh/src/viz_mesh_reports.py` `fig_wireframes()`.

![wireframe matrice4e](outputs/figures/wireframe_matrice4e.png)

**그림 2** — Matrice 4E 메쉬 3면. 같은 코드가 스펙만 바꿔 만든 것이다.

- **왼쪽(shaded)**: 색이 곧 부위(=재질)다 — 셸(body), 프로펠러(prop), 금속 모터(motor),
  짐벌(camera).
- **가운데(wireframe)**: 삼각형 뼈대. 곡면(동체·짐벌)일수록 촘촘하다.
- **오른쪽(top view)**: 위에서 본 로터 배치.

**삼각형 크기 — 세 숫자로 읽는다.** 중앙값만 적으면 최댓값을 못 본다:

| 드론 | 근거 | 꼭짓점 | 삼각형 | 그룹 | 한 변 p50[mm] | p95[mm] | **최대[mm]** | p95/λ | **최대/λ** |
|---|---|---|---|---|---|---|---|---|---|
| Mini 5 Pro | **[B]** | 14,564 | 29,036 | 9 | 3.7 | 7.4 | 94.6 | 0.13 λ | 1.64 λ |
| Matrice 4E | **[A]** | 15,387 | 30,662 | 9 | 6.7 | 12.7 | 157.6 | 0.22 λ | 2.74 λ |

λ 는 우리가 쓰는 최고 대역(WiFi 5.21 GHz)의 파장 **57.5 mm** 다
← 출처: `mesh_verify.json` `meta.lam_hi_mm`·`A_geometry.*.edge_mm`.

**최대값이 λ 를 넘는 것을 어떻게 읽나** — 가장 긴 모서리는 배터리·기판 같은 **평평한 상자**의
모서리다. 평면은 잘게 쪼개도 같은 평면이라 형상이 나빠지지 않고, PO 적분은 면을 λ/11 로
다시 나눠 표본을 뜬다 ← 출처: `src/rcs_po.py` `mesh_to_points`. 그래서 이 최댓값은
위상 표본 문제가 아니다. ⏳ **프로펠러의 삼각형 크기는 별개 축**이고, 기체별 프로펠러 정본화
라운드가 정본이다(§3 끝).

## 2. 왜 인터넷 3D 모델을 그대로 안 쓰나

가장 쉬운 길은 3D 모델 공유 사이트에서 "DJI Mavic 4" 를 검색해 받는 것이다.
그 길을 버린 이유는 둘이다.

1. **치수 미검증** — 취미 모델러가 사진을 보고 눈대중으로 만든 것이 많다. RCS 는 투영 면적과
   세부 형상에 민감해서 치수가 몇 % 틀리면 답이 몇 dB 틀어진다.
2. **라이선스 제약** — 연구 산출물에 재배포 불가·상업 불가 모델을 섞으면 재현 패키지를 공개할 수 없다.

거기에 **레이더 고유의 이유**가 하나 더 붙는다. 게임·렌더용 모델은 겉모습만 그럴듯하면 되지만,
전파의 눈에는 겉껍데기 플라스틱(\|Γ\|=0.28)보다 속의 **배터리·모터 금속**
(\|Γ\|≈1.00)이 훨씬 밝다 — 같은 넓이면 반사 전력이 약 **11 dB**
(≈13배) 차이다
← 출처: `mesh_verify.json` `E_materials.*.gamma_map`(원본 `src/materials.py`, ITU-R P.2040).

⚠ 다만 이 문장을 **«내부 금속이 없으면 속 빈 유령»** 으로 밀어붙이면 지금 상태를 과장한다.
단서 셋을 함께 적어야 한다 — §3.2 에서 숫자로 푼다.

### 2.1 공식 CAD 는 어디까지 있나 — 두 기체는 있고 나머지는 없다

«제조사가 CAD 를 공개하지 않는다» 는 **기체마다 다르다.** 지금 저장소가 가진 것:

| 자료 | 정체 | 축척 검산 |
|---|---|---|
| `assets/meshes/reference/matrice4-M4T_v2.step` | DJI **Matrice 4T** 공식 STEP | 폭 387.501 ↔ 공표 387.5 mm |
| `assets/meshes/reference/WM161_zhankai_1k.glb` | DJI **Mini 2** 공식 3D(펼침) | bbox E1 — 공식 GLB 로 전 상수 실측 — 셸 스테이션이 GLB 와 0.5 % 안 |
| `assets/meshes/reference/x500v2-frame.step` | Holybro **X500 v2** 공식 프레임 STEP | 암 단면 16.0×16.0 mm 재현 |

← 출처: `assets/meshes/reference/SOURCES.md`·`outputs/meshfix_matrice4e.json` `scale_check`·
`outputs/mesh_inspect_body_arms_0816.json` `per_drone.mini2`.

⭐ **그런데 M4T 는 우리 표적이 아니다.** 우리 표적은 Matrice 4**E** 이고 DJI 는 4E 판 CAD 를
공개하지 않는다. 그래서 이 CAD 는 **갈라 써야 한다**
← 출처: `docs/MESH_AUDIT_0816.md` §⑧(사용자 지시로 세운 상시 규칙):

| 부품군 | 4T ↔ 4E | CAD 를 써도 되나 |
|---|---|---|
| 셸(동체)·팔·다리·모터 | 공용 | ✅ 그대로 |
| 어안·비전 센서·비콘 위치 | 공용 | ✅ 그대로 |
| RTK 안테나 | 공용 | ✅ 그대로 |
| ⚠ **짐벌·카메라 블록** | **다르다** — 탑재체가 갈리는 지점 | ⛔ **치수는 쓰지 마라.** 매다는 자리(크래들·댐핑플레이트)만 공용이라 **위치는** 써도 된다 |
| 내부 기판·배터리 | CAD 는 외장 모델이라 애초에 없다 | — |

**나머지 7종에는 공식 CAD 가 없다.** Mavic 4 Pro 도 없다 — 그래서 그 기체의 암 폭은
Mini 5 Pro 실측을 크기비로 옮긴 값이고, 표에 **[C] 계열 유추**로 적혀 있다(§1.3).

⚠ 남은 불확실: «4T 와 4E 의 기체가 공용» 이라는 것은 제원·매뉴얼 대조에서 나온 판단이고
부품 단위로 전수 대조한 것은 아니다 ← 출처: `docs/MESH_AUDIT_0816.md` §⑧ 말미.

### 2.2 그래서 코드로 만든다 — 파라메트릭 CAD 의 4가지 이점

"코드로 만든다" = 치수를 넣으면 형상이 나오는 함수를 짠다는 뜻이다(파라메트릭 CAD).
각 드론은 `DroneSpec` 이라는 데이터클래스 하나로 요약된다 — 대각거리, 무게, 프로펠러
지름·날 수, 로터 수, 공식 외형(L×W×H) … ← 출처: `src/drones.py` `class DroneSpec`.

| 이점 | 설명 | 이 저장소에서 실증된 방식 |
|---|---|---|
| **재현성** | `build_drone(spec)` 한 줄로 같은 메쉬가 다시 나온다 | 메쉬 지문(sha256(정점+삼각형))을 A/B 로 비교해 10종 **비트동일** 확인 |
| **파라메트릭** | 제원이 정정되면 숫자 하나만 고치면 형상 전체가 따라온다 | matrice4e 상수 14건 정정이 한 번에 착지 |
| **부위별 재질** | 만들 때부터 면마다 body/prop/motor … 이름표 → 재질 배정이 공짜 | §3 |
| **버전관리** | 메쉬가 곧 코드니까 형상 변경 이력이 텍스트로 남는다 | 바이너리 3D 파일로는 불가능 |

← 출처: 설계 의도는 `src/drones.py`·`src/geom.py` 모듈 docstring;
지문 A/B 검증은 `outputs/mesh_inspect_body_arms_0816.json` `code_changes.verification`.

**정직성 원칙 — 모르는 값은 모른다고 적는다.** 예컨대 Mini 5 Pro:

> Mini 5 Pro 의 대각(모터-모터 휠베이스)은 DJI 가 공개하지 않는다. 코드는 그 값을 추정으로
> 표시하고, **로터의 실제 위치는 따로 선언한다** — 275 mm 는 암 두께·모터 비례식의 스케일로만 남는다.

← 출처: `src/drones.py` mini5pro `note`. 지금 남아 있는 «모른다» 목록은 §4.4 에 모아 두었다.

### 2.3 ⭐ 참조 자료는 어디까지 제작에 들어갔나

«참조 자료는 채점에만 쓰고 제작에는 안 쓴다» 는 **깔끔하지만 지금은 사실이 아니다.**
참조 자료가 제작에 들어간 자리가 셋 있고, 그것을 감추면 «독립 채점» 이라는 말이 과장이 된다.

| # | 어디에 | 무엇이 들어갔나 | 등급 |
|---|---|---|---|
| ① | **matrice4e 형상 상수 14건** | DJI Matrice 4T 공식 STEP 의 모서리 실측 (접지 −59.82 · 갑판 crown 69.18 · RTK 꼭대기 89.70 · 셸 중심 x 41.71 mm 를 0.1 mm 안에서 재현) | **[A]** |
| ② | **mini2 전 형상 상수** | DJI 공식 GLB(WM161) 실측 — 셸 6 스테이션 중 가운데 4개가 GLB 와 0.5 % 안 | **[A]** |
| ③ | ⏳ **프로펠러 날 법칙** | 실물 참조 프로펠러 측정에서 유도된 상수들 | ⏳ |

← 출처: ①은 `outputs/meshfix_matrice4e.json` + `outputs/mesh_inspect_body_arms_0816.json`
`meshfix_matrice4e_landed`(착지 검증) · ②는 `per_drone.mini2` · ③은 `src/drone_cad.py` 블레이드 법칙 머리말.

⏳ **③ 은 기체별 프로펠러 정본화 라운드가 정본이다.** 이 편에서는 자리만 잡고 값은 적지 않는다 —
그 라운드가 날 시위 분포·기종별 두께·팁 형상을 다시 정하는 중이라, 여기 숫자를 적으면 두 곳이 갈린다.

**그래서 «독립 채점» 이라는 말을 어떻게 써야 하나.** 정확한 문장은 이렇다:

> 채점자 중 **실기체 3D 스캔(Phantom 4, CC-BY)** 은 제작에 한 번도 안 들어갔다 —
> 그 기체에 대해서는 채점이 진짜로 독립이다.
> 반대로 matrice4e·mini2 의 공식 CAD 는 **제작에 들어갔으므로**, 같은 CAD 로 다시 채점한
> 결과는 «맞췄다» 가 아니라 «반영이 착지했다» 로 읽어야 한다.

미리보기 하나: 우리 Phantom 4 메쉬는 실기체 스캔과 표면 거리 중앙값 **4.5 mm**,
90분위 11.7 mm 안에서 겹친다 ← 출처: `mesh_verify.json` `G_scan.scan_to_cad_mm`.
이 기체는 제작에 스캔을 안 썼으므로 이 숫자는 독립 채점이다. 방법과 그림은 mesh08 에서.

## 3. 핵심 원칙 — "OBJ 1개 = 부위 1개 = Sionna 재질 1개"

드론 한 대를 한 덩어리 파일로 저장하지 않고, **부위마다 별도 OBJ 파일**로 저장한다.
이유는 Sionna 의 규칙 때문이다:

> Sionna 는 'OBJ 1개 = SceneObject 1개 = 재질 1개' 이므로, 부위별 재질을
> 주려면 이렇게 부위별로 나눠 저장한다.

← 출처: `src/geom.py` `write_obj_per_group()` docstring (그대로 인용);
같은 원칙이 `README.md` 에 "메쉬 원칙: OBJ 1개 = 부위 1개 = Sionna 재질 1개" 로 선언돼 있다.

함대 전체가 쓰는 부위 그룹은 **13개**다
(기체 하나가 그 전부를 쓰지는 않는다 — 열린 프레임 기체에는 셸이 없고, 접이식에는 데크가 없다):

| 부위(그룹) | 재질 키 | PO \|Γ\| | 쓰는 기체 | 함대 면적 [cm²] | 이 그룹이 무엇인가 |
|---|---|---|---|---|---|
| `body` | `plastic` | 0.28 | 9종 | 13,569 | 동체 셸 |
| `prop` | `prop_plastic` | 0.25 | 10종 | 6,953 | 프로펠러 |
| `battery` | `metal` | 1.00 | 10종 | 5,012 | 배터리팩(내부) — GHz 에서 파우치 포일은 사실상 금속 |
| `camera` | `camera_assembly` | 0.85 | 9종 | 3,553 | 짐벌 카메라(금속 하우징+유리렌즈) |
| `arm` | `carbon` | 0.90 | 3종 | 3,528 | 암 |
| `gear_cf` | `carbon` | 0.90 | 4종 | 2,774 | 카본 튜브 착륙장치 — 'gear'(플라스틱)와 재질이 다르다 |
| `gear` | `plastic` | 0.28 | 10종 | 2,385 | 착륙장치 |
| `pcb` | `pcb` | 0.80 | 10종 | 2,250 | ESC/메인보드(내부) — FR-4 + 구리 그라운드플레인 |
| `motor` | `metal` | 1.00 | 10종 | 2,106 | 모터 |
| `canopy` | `plastic` | 0.28 | 7종 | 1,436 | 상단 캐노피/배터리 |
| `deck` | `carbon` | 0.90 | 1종 | 1,212 | 카본 데크(상·하판 + 스탠드오프) — 셸 없는 열린 프레임 |
| `accent` ⚠ | `plastic` | 0.28 | 4종 | 1,163 | 전방 식별색 |
| `fc` | `pcb` | 0.80 | 1종 | 112 | 비행제어기(Pixhawk 류) — 상판 위 노출 |

← 출처: `outputs/mesh_inspect_materials_check_0816.json` `assignment_audit`
(면적은 10종 합계 실측). ⚠ 표시는 «배정 자체가 다시 봐야 하는 칸» 이다 — §4.4 참조.

### 3.1 같은 재질인데 숫자가 둘이다 — 두 계산 경로가 다른 값을 쓴다

우리는 산란을 두 경로로 계산한다. **Sionna 재질**(광선엔진이 쓰는 (εr, σ) 슬래브)과
**PO 커널의 \|Γ\|**(면적분이 쓰는 실효 반사계수)다. 둘은 **일부러 다른 값**이고,
그 갈림이 얼마인지 적어 두는 것이 정직한 서술이다:

| 재질 키 | Sionna 경로 | Sionna \|Γ\| | PO \|Γ\| | 차이 [dB] | 왜 다른가 |
|---|---|---|---|---|---|
| `camera_assembly` | ITU metal | 0.99980 | 0.85000 | -1.41 | 짐벌 카메라 = **금속 하우징 + 유리 렌즈 + 짐벌 모터** |
| `pcb` | ITU metal | 0.99980 | 0.80000 | -1.94 | ESC/메인보드 = FR-4 유전체 + **구리 그라운드플레인** |
| `plastic` | custom (εr,σ) 슬래브 | 0.24369 | 0.28000 | +1.21 | 드론 셸(ABS/PC) |
| `plastic_blue` | custom (εr,σ) 슬래브 | 0.24369 | 0.28000 | +1.21 | 파란 모서리 트림 — 전파물성은 plastic 과 동일(색만 다름) |
| `prop_plastic` | custom (εr,σ) 슬래브 | 0.24369 | 0.25000 | +0.22 | 프로펠러 — **재질은 plastic 과 동일**(같은 ABS/PC·εr 2 |
| `carbon` | custom (εr,σ) 슬래브 | 0.98867 | 0.90000 | -0.82 | 탄소섬유(도전성) — ITU 에 없음 |

← 출처: `outputs/mesh_inspect_materials_check_0816.json` `engine_divergence`
(fc = 3.5 GHz, 수직입사).

**왜 갈리나** — Sionna 쪽은 «반무한 벌크» 프레넬 값이고, PO 쪽은 «얇은 판의 앞뒷면 간섭까지
넣은 실효값» 이다. 드론 셸은 두께 **0.75 mm** 급이라 그 차이가 실재한다
← 출처: `docs/MATERIAL_CORRECTION.md`(셸 정본 두께 = DJI 공식 CAD 벽 두께 실측의 중앙값).

⚠ **현재의 한계** — 우리 PO 커널에는 **두께라는 개념이 없다.** \|Γ\| 하나를 상수로 받는다.
즉 PO 경로는 «두꺼운 판» 극한으로 계산하고, 두께는 그 상수를 고를 때 한 번만 반영된다.
이 한계는 지금 그대로 있고, 값은 발표까지 동결돼 있다.

### 3.2 «내부 금속이 없으면 속 빈 유령» — 방향은 맞고, 단서가 셋 있다

§2 의 문장을 지금 상태로 정확히 다시 쓴다.

**단서 ① — 우리 배터리는 상한값이다.** 팩 **외피 6면 전부**를 금속으로 둔다. 실물 팩은
플라스틱 케이스 안에 셀 스택이 들어 있어, 되쏘는 금속면은 더 작다. 크기는 1~3 dB 급이고
**미해결로 선언돼 있다** ← 출처: `outputs/mesh_inspect_internal_metal_0816.json` `battery_material`.

**단서 ② — 그 «내부» 금속이 실제로 셸 안에 있는 기체는 소수다.**

| 기체 | 판정 | 무엇을 뜻하나 |
|---|---|---|
| Mini 5 Pro | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Mavic 4 Pro | **PASS** | 금속 상자가 전부 셸 안에 있다 |
| Matrice 4E | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| S1000+ | **N/A** | 설계상 열린 프레임이라 «셸 안» 이라는 물음이 성립하지 않는다 |
| Phantom 4 | **PASS** | 금속 상자가 전부 셸 안에 있다 |
| Typhoon H (H480) | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| X500 V2 | **N/A** | 설계상 열린 프레임이라 «셸 안» 이라는 물음이 성립하지 않는다 |
| Phantom 3 Professional | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Matrice 350 RTK | **FAIL** | 금속 상자의 일부가 셸 **밖**으로 나와 있다 |
| Mini 2 | **UNKNOWN** | 셸이 닫혀 있지 않아 안/밖 판정 자체가 정의되지 않는다 |

← 출처: `outputs/mesh_internal_metal_check_0816.json`.
이것이 왜 중요한가: 우리 SBR 경로는 **셸을 맞은 광선만** 내부를 투과로 본다. 금속 상자가
셸 밖으로 나와 있으면 그 상자는 «내부 산란체» 가 아니라 그냥 겉면이 된다.

**단서 ③ — 카메라 조립품의 \|Γ\|=0.85 는 출처가 없다.** 저장소가 스스로 그렇게 적는다
← 출처: `docs/MATERIAL_SOURCES.md` §6-4. 그 값을 유전체로 바꿔 보면 방위평균 σ 가
el 0/−30/−60° 에서는 −2.2…+0.1 dB 움직이는데, **바로 아래(나디르, el −90°)에서는
−6.5…+2.7 dB** 로 훨씬 크게 움직인다
← 출처: `outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary.gate_D_dielectric_swing_db`.
원인은 짐벌을 매다는 **방진판이 수평 평판**이라 바로 아래 방향에 정반사가 서기 때문이다.

⇒ **절대 σ 를 인용할 때는 «배터리는 팩 외피 전체를 금속으로 본 상한값» 과**
**«카메라 0.85 는 출처 없는 값» 을 함께 적어야 한다.**

### 3.3 ⏳ 프로펠러 — 이 절은 기체별 프로펠러 정본화 결과로 채운다

> ⏳ **이 절은 기체별 프로펠러 정본화 결과로 채운다.** 날 시위 분포·기종별 두께·팁 형상·λ 대비 삼각형 크기는 그 라운드가 정본이므로, 이 편에서 값을 적으면 두 곳이 갈린다.

지금 확실한 것만 적는다.

- **움직이는 성분(AC)은 정의상 프로펠러만 남는다.** 정지한 동체의 기여는 순수 DC 라   DC 를 걷어내면 사라진다. 이것은 측정이 아니라 구조적 필연이다.
- **총 반사(σ)로는 동체가 훨씬 세다** — matrice4e 부품 분해에서 프로펠러 몫은   el −30° 에서 2.4 %(16.2 dB 아래), el 0° 에서 0.2 %(28.1 dB 아래)다.
- ⇒ «프로펠러가 표적을 지배한다» 는 **AC 채널 한정 문장**이다. 총 σ 문장으로 옮겨 쓰면 틀린다.

← 출처: 부품 분해는 `docs/MESH_AUDIT_0816.md` §④-1(우리 PO 커널로 matrice4e 를 부품별로 분해).

덤으로 **분절(articulated) 자세**도 부위별 OBJ 에서 공짜로 얻는다: 몸체와 프로펠러가 별개
조각이라 몸체 기울기와 로터별 회전 위상을 따로 줄 수 있다
← 출처: `src/drones.py` `pose_articulated()` docstring.

## 4. 전체 지도 — 자료 → 제작 → 검사기 → 원장

![pipeline map](outputs/figures/pipeline_map.png)

**그림 3** — 모든 정보가 어디서 와서 어떻게 채점되는지 한 장 지도
← 그림 생성: `report_mesh/src/viz_mesh_reports.py` `fig_pipeline_map()`.

층이 **넷**이다. 앞의 세 층은 예전부터 있었고, 넷째(원장)를 따로 세운 것이 지금 구조다.

**① 자료층** — 세 종류, 역할이 다르다:

- **제조사 공식 제원표** → `docs/drone_research.json` → `docs/SPECS.md`. **모든 기체의 치수 출발점.**
- **제조사 공식 CAD** (Matrice 4T STEP · Mini 2 GLB · X500 v2 STEP) — **제작에도 들어간다**(§2.3).
- **실기체 3D 스캔 / 타사 실물 CAD** — 채점 전용.

**② 제작층** — 숫자가 형상이 되는 곳:

- `src/drones.py` 의 `DroneSpec` 10개 → `src/drone_cad.py` + `src/cadkit.py` 가
  드론을 **cad(trimesh+manifold3d)** 엔진으로 깎는다(드론 제작 경로는 이 하나뿐이다)
  ← 출처: `mesh_verify.json` `meta.mesh_engine`.
- **왜 trimesh + manifold3d 인가?** 프리미티브를 그냥 겹쳐 놓으면 겹친 파트의 **내부에 숨은 면**이
  표면 데이터에 그대로 남고, PO 는 그런 면까지 반사면으로 센다. manifold3d 의 **불리언 합집합**은
  겹친 파트를 한 덩어리로 녹여 내부 면이 애초에 존재하지 않게 한다
  ← 출처: `src/drone_cad.py` 머리말 "왜 이게 RCS 에 중요한가".
  (자작 `geom.Mesh` 의 역할은 **컨테이너와 무대**다: 완성 메쉬 담기(.v/.f/.g)·부위별 OBJ 저장,
  그리고 범용 프리미티브 제작.)
- 완성 메쉬는 `write_obj_per_group()` 으로 **부위별 OBJ** 저장(§3) → `src/materials.py` 가
  부위→전파재질 배정.

**③ 검사기층 · ④ 원장층** — §4.2~§4.3.

### 4.2 검사기는 하나가 아니다 — 다섯이고, 역할이 다르다

| 검사기 | 언제 도나 | 무엇을 보나 | 범위·단서 |
|---|---|---|---|
| `src/cadkit.py` `Assembly.check` | 빌드 도중 | 파트 하나를 붙일 때마다 | 부품 단위 수밀·법선 |
| `src/mesh_check.py` | 출하 게이트 | 10 검사 + 예산표 | `python src/drones.py`(OBJ 내보내기) 한 문에 배선. `MESH_GATE=off` 로 끌 수 있고, RCS·렌더가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** |
| `report_mesh/src/verify_mesh_suite.py` | 원장 생성 | A~I 9절 | 이 시리즈의 숫자를 만든다. I 절(SBR)만 GPU |
| `benchmark/check_gimbal_sensors_0816.py` | 특수 검사 | 짐벌·센서 게이트 A~D | 부착·삼킴·선언초과·재질 민감도 |
| `benchmark/mesh_internal_metal_check.py` | 특수 검사 | 내부 금속 포함 판정 | «금속 상자가 정말 셸 안인가» |

← 출처: 각 파일의 모듈 docstring·`__main__` 배선.

**게이트의 범위를 정확히 적는다.** `src/mesh_check.py` 의 회귀 게이트는
`python src/drones.py`(부위별 OBJ 내보내기) **한 문**에 걸려 있다. 세 가지 단서가 있다:

1. 환경변수 `MESH_GATE=off` 로 끌 수 있다.
2. RCS·렌더·마이크로도플러가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** —
   전 기종 검사가 수십 초 걸려서 import 시점에 걸지 않는다고 코드가 스스로 적는다.
3. 그래서 «메쉬를 쓰는 모든 경로가 검사를 통과한다» 고 쓰면 지금 상태보다 강한 말이 된다.

**원장층(④)** — 검사 결과가 모이는 파일들이다. 이 편 머리의 «무엇을 근거로 하는가» 표가
그 목록이고, 이 노트북의 모든 숫자가 거기서 나왔다.

### 4.3 무엇을 검사하나 — 10 검사, 그리고 «예산» 이라는 규약

| # | 검사 | 무엇을 묻나 |
|---|---|---|
| 1 | **수밀(watertight)** | 부품이 닫힌 껍질인가 |
| 2 | **경계 모서리** | 삼각형 하나만 쓰는 모서리 = 구멍의 테두리. **원칙은 0**, 예산으로만 예외 |
| 3 | **winding** | 이웃한 면이 같은 방향으로 감겼는가 |
| 4 | **법선 방향** | 닫힌 부품의 부호있는 부피가 양수인가 |
| 5 | **부호부피(원본 인덱스)** | trimesh 를 **전혀 안 거치고** 출하 인덱스에서 손계산 |
| 6 | **퇴화면 — 절대 + 상대** | 면적 잣대에 더해 **최소 내각 <0.5°** 슬리버를 센다 |
| 7 | **그룹 안 겹침** | 같은 그룹의 두 부품이 서로 파묻혔는가(PO 면적 이중계상) |
| 8 | **치수 대조** | 프롭 지름·로터 대각·공표 외형을 `DroneSpec` 의 수와 대조 |
| 9 | **손대칭성** | 로터별 날 비틀림 방향이 회전방향과 맞는가 — **거울상 기체 탐지** |
| 10 | **프롭↔모터 벨 관통** | 원통 근사(빠름) + 솔리드 내부판정(판정 기준) |

← 출처: `src/mesh_check.py` 모듈 docstring·구현.

⭐ **«통과» 의 뜻이 «0» 이 아니다.** 검사기는 항목마다 **예산 표**를 들고 있고,
통과란 «선언된 예산 안» 이라는 뜻이다:

| 예산 | 지금 값 | 무엇을 뜻하나 |
|---|---|---|
| `BOUNDARY_EDGE_BUDGET` | 기본 **0**, 예외 (mini2, body) = 3 | 구멍은 원칙적으로 없어야 한다. 예외 하나가 명시적으로 선언돼 있다 |
| `SLIVER_BUDGET` | 기종별 198~638 | 아주 뾰족한 삼각형 개수. 면적 비중은 0.0001~0.03 % 라 σ 에는 무해하고, 감시하는 이유는 법선이 수치적으로 불안정한데 PO 조명 판정이 `n̂·û>0` 이기 때문이다 |
| `GROUP_OVERLAP_BUDGET_PCT` | 기본 0.1 %, battery 4종 55 % | 같은 그룹 안에서 부품이 파묻힌 비율 |
| `DIM_TOL_PCT` | 프롭 지름 1 % · 외형 1 % · 대각 3 % (mini5pro 예외 12 %) | 공표 숫자와의 허용 오차 |
| `HANDEDNESS_MIN_ABS` | 0.05 | 날 비틀림 지표의 최소 크기. 이보다 작으면 «비틀리지 않았다» 는 뜻이라 부호를 믿을 수 없다 |

← 출처: `src/mesh_check.py` 예산 표.

**예산 표를 왜 이렇게 쓰나** — 이 표들은 «이만큼이 옳다» 가 아니라 **«지금 이만큼이다» 라는**
**선언**이다. 값은 전수 실측으로 채웠고 여유는 약 10 % 다. 그래서 **새로 생기는 결함은**
**예산을 넘겨 실패한다.** 숨기지 않으면서도 회귀를 막는 방식이다.

**이 검사가 아직 못 보는 것** — 정직하게 남긴다:

- **동일평면 겹침** — 두 부품 표면이 정확히 같은 자리에 있으면 관통 검사가 못 본다. 9기체에서 34쌍이 그 상태다(가장 큰 것은 s1000plus body↔battery 16,745.8 mm²).
- ⏳ **삼각형 크기를 파장에 묶는 규약** — 프로펠러 축은 기체별 프로펠러 정본화 라운드가 맡는다.
- **기종별 재질 분기** — `drone_gamma_map(spec, fc)` 이 `spec` 을 안 쓴다. 지금은 재질이 기체와 무관해서 맞지만, 기종별 재질이 생기는 순간 조용히 틀린 답을 준다.
- **PO 경로의 가림** — `rcs_po.py` 가 자기 docstring 에서 자기차폐·다중반사를 무시한다고 선언한다. 부품 속에 묻힌 면이 그 경로에서는 이중계상된다(재질 가중으로 +0.03~+0.69 dB).

## 4.4 지금 남은 결함 — 있는 그대로

아래는 **현재 메쉬가 안고 있는 어긋남**이다. 크기를 함께 적어, 어느 결론이 흔들리고
어느 결론이 안 흔들리는지 독자가 직접 판단할 수 있게 한다.

| 무엇 | 기체 | 지금 이만큼 | 어디에 실리나 |
|---|---|---|---|
| 공표 높이를 **형상이 아니라 세로 배율**로 맞춘다 | mini5pro · mavic4pro | 세로 배율 1.2985 / 1.3524 — 형상표의 셸 높이 45.05 / 62.10 mm 가 메쉬에서 59.99 / 87.70 mm 로 나온다 | 평판극한 σ 상한 +2.27 / +2.62 dB (방위평균, el 0°) |
| 짐벌이 착륙발보다 아래 | mavic4pro | 카메라 최저점이 발보다 15.35 mm 아래. 발을 바닥으로 놓고 같은 규칙을 풀면 세로 배율이 1.3524 → 1.5977 (18.14 %) | 위 세로 배율의 **원인** — 예산 구멍을 가린다 |
| 뜬 파트(기체에 안 닿는 부품) | phantom4 · phantom3 · m350rtk · x500v2 | 착륙아치 8.3~8.5 / 13.7~13.8 mm · 프롭 허브 6.0 mm · 레일 4.0 mm | 간극 0.05~0.16 λ @3.5 GHz — 면적은 그대로고 가림·다중반사·위상이 바뀐다 |
| L/W 강제가 남아 축간거리가 부푼다 | phantom4 | 공표 350 → 메쉬 356.92 mm (+1.98 %) | 평판극한 −0.39 dB (el 0°) |
| 로터면이 공식 CAD 보다 위 | matrice4e | 18.5 mm = 0.216 λ @3.5 GHz. 명세가 F19~F21 로 «엔진 변경 필요» 라 미뤄 둔 자리 | 프롭 장착 높이가 함께 움직인다 ⏳ |
| 셸에 삼각형 1장 구멍 | mini2 | 경계 모서리 3개, 구멍 넓이 약 0.35 mm² (λ²/21000) | σ 는 무시할 수준. 진짜 피해는 **안/밖 판정이 정의되지 않는 것** |
| 카본 판이 `plastic` 그룹에 있다 | s1000plus | 판 2장만 세도 body 합집합 전 면적의 69.3 % (스탠드오프 기둥까지 넣은 스택은 74.5 %) | 면 반사율 +10.14 dB (carbon 0.90 ↔ plastic 0.28) |

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings`·
`outputs/mesh_inspect_gimbal_sensors_0816.json` `_summary`·
`outputs/mesh_inspect_materials_check_0816.json` `findings`.

**dB 를 읽는 법** — 위 표의 dB 는 대부분 **평판극한 상한**이다. «같은 크기 평판이라면 최대
이만큼» 이라는 뜻이지 커널이 계산한 σ 가 아니다. 크기 감각을 주는 자로만 쓸 것.

### 지금 «모른다» 고 선언한 것

- mavic4pro 의 세로 예산 35.2 mm 가 **어디에** 있어야 하는지 못 정했다. 공표 언폴드 높이 135.2 mm 는 공식이지만 그 135.2 를 다리·셸·짐벌·모터에 어떻게 나누는지는 사진 한 장으로 안 풀린다 — matrice4e 처럼 공식 CAD 가 필요하고 DJI 는 Mavic 4 Pro CAD 를 공개하지 않는다.
- mini5pro 셸 높이의 1차 출처가 없다. `fh = 0.495` 는 공표 높이(91, **프롭 포함**)에 대한 비율이고, 그 91 자체가 프롭을 포함하므로 셸 높이를 직접 구속하지 않는다. 폴디드 68 mm 로 교차검산하려면 짐벌 매달림 길이를 따로 재야 하는데 그 값도 실측이 없다.
- B3 의 «기수 정면 정반사 10 dB» 는 평판극한 상한일 뿐 커널 결과가 아니다. 진짜 값을 알려면 스무딩 0/4 두 메쉬로 PO 를 돌려야 하는데 이 라운드는 σ 파일을 열지 않았다.
- phantom3·phantom4 착륙아치가 «어디에» 붙어야 하는지 — 매뉴얼 정면도가 붙는 곳 좌우 스팬은 주지만 앞뒤 부착점은 셸 곡면과의 교선이라 표에서 못 읽는다.
- x500v2 배터리 트레이 2.65 mm 는 2026-08-04 원장이 «면-대-면 접촉의 거짓양성» 이라 적었는데 양방향 잣대로도 남는다. 어느 쪽이 맞는지 판정하지 않았다.
- ⏳ 프로펠러 축 전부 — 날 시위·두께·비틀림·팁·기체별 정본화는 다른 라운드가 맡는다. 이 파일은 프롭 장착 높이만 인계용으로 적었다.
- **배터리 재질** — ⚠ **미해결로 선언한다.** 지금 고치지 않는 이유: 셀 스택의 실제 치수가 1차 출처 0 이고, 추정으로 줄이면 «측정 아닌 값» 을 또 하나 심는다. 대신 **모든 절대 σ 인용에 «배터리는 팩 외피 전체를 금속으로 본 값(상한 쪽 1~3 dB)» 단서를 붙일 것.**
- **카메라 재질 0.85 의 출처** — docs/MATERIAL_SOURCES.md §6-4 가 이미 «출처 없음 · 총 σ 를 최대 1.81 dB 움직임» 으로 적어 뒀다 (그 1.81 은 mavic4pro·1.843 GHz 한 팔의 값이다).

⭐ **빈칸이 가짜 값보다 낫다.** 위 항목들은 값을 채워 넣는 대신 비워 두었다.

## 5. 시리즈 목차 — mesh02~08

| 편 | 주제 (한 줄) |
|---|---|
| **mesh02** | 도구 상자 — 어떤 파이썬 라이브러리를 왜 골랐나, 그리고 검사기는 무엇을 보고 무엇을 못 보나 |
| **mesh03** | 자료 수집 — 모든 숫자·모델의 출처(공식 제원·공식 CAD·실기체 스캔·타사 CAD, 라이선스) |
| **mesh04** | 몸체 CAD — 스펙 숫자가 드론 모양이 되기까지(로프트·조립 순서·기종별 개성) |
| **mesh05** | 프로펠러 — ⏳ 기체별 프로펠러 정본화 라운드가 정본 |
| **mesh06** | 색이 곧 재질 — 부위별 전파 재질과 두 계산 경로의 \|Γ\| |
| **mesh07** | 검증 ① 기하 — 수밀·법선·삼각형 품질·대칭·부위 겹침 |
| **mesh08** | 검증 ② 실물·물리 — 치수 대조·실기체 스캔 chamfer·수치 수렴 |

모든 편이 이 편과 같은 규약을 따른다: 생성물 노트북, 수치는 원장에서 주입, 사실마다 `← 출처:`.

**이 시리즈의 지위** — `README.md` 편성에서 report_mesh 8편은 **부록**이다. 본편 쪽에는
같은 주제의 별편 **2-3 «표적을 짓는다 — 메쉬와 재질»**(`reports/02_3_target-mesh.ipynb`)이
따로 있다. 둘의 차이는 독자다 — 별편 2-3 은 결론을 쓰고, 이 시리즈는 **만드는 법과 채점 방법**을 쓴다.

## 재현 명령

```bash
PY=/workspace/.venvs/py312/bin/python
cd /workspace/sionna

# 1) 원장 재생성 — 이것부터. I 절(SBR)만 GPU 가 필요하다.
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py             # 전체
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py --skip-sbr  # GPU 없이 A~H

# 2) 그림 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/viz_mesh_reports.py

# 3) 노트북 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/make_mesh01.py
```

⭐ **순서를 지켜야 한다.** 생성기는 원장이 레지스트리와 다르면 **일부러 멈춘다** —
«원장 10종 vs 레지스트리 N종» 으로 예외를 던진다
← 출처: `report_mesh/src/mesh_ledger.py` `ledger_order()`.
리포트가 틀린 개수를 조용히 쓰는 것보다 멈추는 편이 낫다는 규약이다.


⚠ **지금 원장의 I 절(SBR)은 이월된 값이다** — `--skip-sbr` 로 돌린 갱신이 GPU 증거를
지우지 않게 직전 원장에서 옮겨 왔고, `stale: true` 로 표시돼 있다. 다른 절과 세대가
다르므로 면 수·σ 를 나란히 인용하지 말 것.

---

**다음 편** → [mesh02 — 도구 상자](mesh02_tools.ipynb) : 어떤 파이썬 라이브러리를 왜 골랐고,
검사기가 무엇을 보고 무엇을 못 보는지.